# 03: Exploratory Data Analysis (EDA)

**Objective**: Identify the key demographic, clinical, and treatment patterns that drive 30-day hospital readmissions in diabetic patients across 130 US hospitals.

**KPIs Under Investigation**:
- 30-Day Readmission Rate (%)
- Average Length of Stay (Days)

## Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

## Load Processed Data

In [ ]:
df = pd.read_csv('../data/processed/cleaned_diabetic_data.csv')
print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

---
## Part 1: Understanding the Target Variable
Before diving into predictors, we must understand the distribution of our primary KPI — the 30-day readmission rate.

In [ ]:
readmit_counts = df['Readmission Status'].value_counts()
readmit_pcts = df['Readmission Status'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0].bar(readmit_counts.index, readmit_counts.values, color=colors)
axes[0].set_title('Readmission Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Number of Encounters')
for i, (v, p) in enumerate(zip(readmit_counts.values, readmit_pcts.values)):
    axes[0].text(i, v + 500, f'{v:,}\n({p:.1f}%)', ha='center', fontweight='bold')

axes[1].pie(readmit_counts.values, labels=readmit_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=[0, 0, 0.05])
axes[1].set_title('Readmission Proportion', fontweight='bold')

plt.tight_layout()
plt.show()

rate_30d = df['Readmitted Under 30 Days'].mean() * 100
print(f'\n📊 Overall 30-Day Readmission Rate: {rate_30d:.2f}%')

---
## Part 2: Demographic Analysis
Identifying which patient segments are most vulnerable to early readmission.

### 2.1 Age vs Readmission Rate

In [ ]:
age_readmit = df.groupby('Age').agg(
    total_encounters=('Readmitted Under 30 Days', 'count'),
    readmit_rate=('Readmitted Under 30 Days', 'mean')
).reset_index()
age_readmit['readmit_rate'] = age_readmit['readmit_rate'] * 100

fig, ax1 = plt.subplots(figsize=(12, 6))
bars = ax1.bar(age_readmit['Age'].astype(str), age_readmit['total_encounters'],
               color='#3498db', alpha=0.6, label='Total Encounters')
ax1.set_xlabel('Age (Midpoint)', fontweight='bold')
ax1.set_ylabel('Number of Encounters', color='#3498db')

ax2 = ax1.twinx()
ax2.plot(age_readmit['Age'].astype(str), age_readmit['readmit_rate'],
         color='#e74c3c', marker='o', linewidth=2.5, label='30-Day Readmission Rate (%)')
ax2.set_ylabel('Readmission Rate (%)', color='#e74c3c')

plt.title('Age Distribution & 30-Day Readmission Rate', fontsize=14, fontweight='bold')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.95))
plt.tight_layout()
plt.show()

### 2.2 Race vs Readmission Rate

In [ ]:
race_readmit = df.groupby('Race')['Readmitted Under 30 Days'].agg(['mean', 'count']).reset_index()
race_readmit.columns = ['Race', 'Readmit Rate', 'Count']
race_readmit['Readmit Rate'] = race_readmit['Readmit Rate'] * 100
race_readmit = race_readmit.sort_values('Readmit Rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(race_readmit['Race'], race_readmit['Readmit Rate'], color='#9b59b6')
ax.set_xlabel('30-Day Readmission Rate (%)')
ax.set_title('Readmission Rate by Race', fontsize=14, fontweight='bold')
for bar, count in zip(bars, race_readmit['Count']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}% (n={count:,})', va='center')
plt.tight_layout()
plt.show()

### 2.3 Gender vs Readmission Rate

In [ ]:
gender_readmit = df.groupby('Gender')['Readmitted Under 30 Days'].agg(['mean', 'count']).reset_index()
gender_readmit.columns = ['Gender', 'Readmit Rate', 'Count']
gender_readmit['Readmit Rate'] = gender_readmit['Readmit Rate'] * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(gender_readmit['Gender'], gender_readmit['Readmit Rate'],
              color=['#e74c3c', '#3498db'], width=0.5)
ax.set_ylabel('30-Day Readmission Rate (%)')
ax.set_title('Readmission Rate by Gender', fontsize=14, fontweight='bold')
for bar, count in zip(bars, gender_readmit['Count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.2f}%\n(n={count:,})', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Admission & Discharge Pattern Analysis
Understanding how the nature of admission and discharge destination relates to readmission.

### 3.1 Admission Type vs Readmission

In [ ]:
admit_readmit = df.groupby('Admission Type')['Readmitted Under 30 Days'].agg(['mean', 'count']).reset_index()
admit_readmit.columns = ['Admission Type', 'Readmit Rate', 'Count']
admit_readmit['Readmit Rate'] = admit_readmit['Readmit Rate'] * 100
admit_readmit = admit_readmit[admit_readmit['Count'] > 100].sort_values('Readmit Rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(admit_readmit['Admission Type'], admit_readmit['Readmit Rate'], color='#e67e22')
ax.set_xlabel('30-Day Readmission Rate (%)')
ax.set_title('Readmission Rate by Admission Type', fontsize=14, fontweight='bold')
for bar, count in zip(bars, admit_readmit['Count']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}% (n={count:,})', va='center')
plt.tight_layout()
plt.show()

### 3.2 Primary Diagnosis Group vs Readmission

In [ ]:
diag_readmit = df.groupby('Primary Diagnosis Group')['Readmitted Under 30 Days'].agg(['mean', 'count']).reset_index()
diag_readmit.columns = ['Diagnosis Group', 'Readmit Rate', 'Count']
diag_readmit['Readmit Rate'] = diag_readmit['Readmit Rate'] * 100
diag_readmit = diag_readmit[diag_readmit['Count'] > 500].sort_values('Readmit Rate', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(diag_readmit['Diagnosis Group'], diag_readmit['Readmit Rate'],
               color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(diag_readmit))))
ax.set_xlabel('30-Day Readmission Rate (%)')
ax.set_title('Readmission Rate by Primary Diagnosis Group', fontsize=14, fontweight='bold')
for bar, count in zip(bars, diag_readmit['Count']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}% (n={count:,})', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 4: Clinical Feature Deep-Dive
Analyzing the hospital encounter metrics that are most strongly tied to readmission outcomes.

### 4.1 Length of Stay (KPI 2) Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Days in Hospital'].hist(bins=14, ax=axes[0], color='#1abc9c', edgecolor='white')
axes[0].set_title('Distribution of Days in Hospital', fontweight='bold')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['Days in Hospital'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["Days in Hospital"].mean():.1f} days')
axes[0].legend()

los_readmit = df.groupby('Days in Hospital')['Readmitted Under 30 Days'].mean() * 100
axes[1].plot(los_readmit.index, los_readmit.values, marker='o', color='#e74c3c', linewidth=2)
axes[1].set_title('Readmission Rate vs Length of Stay', fontweight='bold')
axes[1].set_xlabel('Days in Hospital')
axes[1].set_ylabel('30-Day Readmission Rate (%)')
axes[1].fill_between(los_readmit.index, los_readmit.values, alpha=0.15, color='#e74c3c')

plt.tight_layout()
plt.show()

print(f'📊 Average Length of Stay: {df["Days in Hospital"].mean():.2f} days')
print(f'📊 Median Length of Stay: {df["Days in Hospital"].median():.1f} days')

### 4.2 Prior Inpatient Visits — The Strongest Predictor

In [ ]:
inpatient_readmit = df.groupby('Prior Inpatient Visits').agg(
    readmit_rate=('Readmitted Under 30 Days', 'mean'),
    count=('Readmitted Under 30 Days', 'count')
).reset_index()
inpatient_readmit['readmit_rate'] = inpatient_readmit['readmit_rate'] * 100
inpatient_readmit = inpatient_readmit[inpatient_readmit['count'] >= 50]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(inpatient_readmit['Prior Inpatient Visits'].astype(str),
       inpatient_readmit['readmit_rate'], color='#c0392b')
ax.set_xlabel('Number of Prior Inpatient Visits (Past Year)')
ax.set_ylabel('30-Day Readmission Rate (%)')
ax.set_title('Prior Inpatient Visits vs 30-Day Readmission Rate', fontsize=14, fontweight='bold')
for i, row in inpatient_readmit.iterrows():
    ax.text(str(row['Prior Inpatient Visits']), row['readmit_rate'] + 0.5,
            f'{row["readmit_rate"]:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 4.3 Number of Emergency Visits vs Readmission

In [ ]:
emerg_readmit = df.groupby('Prior Emergency Visits').agg(
    readmit_rate=('Readmitted Under 30 Days', 'mean'),
    count=('Readmitted Under 30 Days', 'count')
).reset_index()
emerg_readmit['readmit_rate'] = emerg_readmit['readmit_rate'] * 100
emerg_readmit = emerg_readmit[emerg_readmit['count'] >= 50]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(emerg_readmit['Prior Emergency Visits'].astype(str),
       emerg_readmit['readmit_rate'], color='#d35400')
ax.set_xlabel('Number of Prior Emergency Visits (Past Year)')
ax.set_ylabel('30-Day Readmission Rate (%)')
ax.set_title('Prior Emergency Visits vs 30-Day Readmission Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.4 Number of Medications & Lab Procedures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

readmitted = df[df['Readmitted Under 30 Days'] == 1]
not_readmitted = df[df['Readmitted Under 30 Days'] == 0]

axes[0].hist(not_readmitted['Number of Medications'], bins=40, alpha=0.6,
             color='#2ecc71', label='Not Readmitted (<30d)', density=True)
axes[0].hist(readmitted['Number of Medications'], bins=40, alpha=0.6,
             color='#e74c3c', label='Readmitted (<30d)', density=True)
axes[0].set_title('Medication Count Distribution', fontweight='bold')
axes[0].set_xlabel('Number of Medications')
axes[0].legend()

axes[1].hist(not_readmitted['Lab Procedures'], bins=40, alpha=0.6,
             color='#2ecc71', label='Not Readmitted (<30d)', density=True)
axes[1].hist(readmitted['Lab Procedures'], bins=40, alpha=0.6,
             color='#e74c3c', label='Readmitted (<30d)', density=True)
axes[1].set_title('Lab Procedures Distribution', fontweight='bold')
axes[1].set_xlabel('Number of Lab Procedures')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Part 5: Medication & Treatment Impact
Analyzing whether changes in diabetic medication during the encounter affect readmission outcomes.

### 5.1 Medication Change vs Readmission

In [ ]:
med_change = df.groupby('Medication Changed')['Readmitted Under 30 Days'].agg(['mean', 'count']).reset_index()
med_change.columns = ['Medication Changed', 'Readmit Rate', 'Count']
med_change['Readmit Rate'] = med_change['Readmit Rate'] * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(med_change['Medication Changed'], med_change['Readmit Rate'],
              color=['#27ae60', '#e74c3c'], width=0.4)
ax.set_ylabel('30-Day Readmission Rate (%)')
ax.set_title('Impact of Medication Change on Readmission', fontsize=14, fontweight='bold')
ax.set_xticklabels(['No Change', 'Changed'])
for bar, row in zip(bars, med_change.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{row[2]:.2f}%\n(n={row[3]:,})', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Insulin Status vs Readmission

In [ ]:
insulin_readmit = df.groupby('Insulin')['Readmitted Under 30 Days'].agg(['mean', 'count']).reset_index()
insulin_readmit.columns = ['Insulin Status', 'Readmit Rate', 'Count']
insulin_readmit['Readmit Rate'] = insulin_readmit['Readmit Rate'] * 100
insulin_readmit = insulin_readmit.sort_values('Readmit Rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(insulin_readmit['Insulin Status'], insulin_readmit['Readmit Rate'],
              color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'])
ax.set_ylabel('30-Day Readmission Rate (%)')
ax.set_title('Insulin Dosage Status vs 30-Day Readmission', fontsize=14, fontweight='bold')
for bar, count in zip(bars, insulin_readmit['Count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.2f}%\n(n={count:,})', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 5.3 A1C Result and Glucose Serum vs Readmission

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

a1c_readmit = df.groupby('A1C Result')['Readmitted Under 30 Days'].mean() * 100
a1c_readmit.plot(kind='bar', ax=axes[0], color='#8e44ad')
axes[0].set_title('A1C Result vs Readmission Rate', fontweight='bold')
axes[0].set_ylabel('30-Day Readmission Rate (%)')
axes[0].tick_params(axis='x', rotation=0)

glu_readmit = df.groupby('Glucose Serum Test')['Readmitted Under 30 Days'].mean() * 100
glu_readmit.plot(kind='bar', ax=axes[1], color='#2980b9')
axes[1].set_title('Glucose Serum Test vs Readmission Rate', fontweight='bold')
axes[1].set_ylabel('30-Day Readmission Rate (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

---
## Part 6: Correlation Analysis
Identifying the strongest linear relationships between numerical features and the 30-day readmission target.

In [ ]:
numeric_cols = ['Age', 'Days in Hospital', 'Lab Procedures', 'Non-Lab Procedures',
                'Number of Medications', 'Prior Outpatient Visits', 'Prior Emergency Visits',
                'Prior Inpatient Visits', 'Total Diagnoses', 'Readmitted Under 30 Days']

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix: Clinical Features vs 30-Day Readmission', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Top Correlations with 30-Day Readmission

In [ ]:
target_corr = corr_matrix['Readmitted Under 30 Days'].drop('Readmitted Under 30 Days').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.set_xlabel('Correlation Coefficient')
ax.set_title('Feature Correlation with 30-Day Readmission (Ranked)', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

---
## Part 7: Multi-Readmission Comparison
Comparing the clinical profiles of patients who were readmitted under 30 days, after 30 days, and those who were never readmitted.

In [ ]:
comparison_cols = ['Days in Hospital', 'Number of Medications', 'Lab Procedures',
                   'Prior Inpatient Visits', 'Prior Emergency Visits', 'Total Diagnoses']

comparison = df.groupby('Readmission Status')[comparison_cols].mean()
comparison = comparison.reindex(['<30', '>30', 'NO'])
comparison.index = ['Under 30 Days', 'After 30 Days', 'Not Readmitted']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
colors = ['#e74c3c', '#f39c12', '#2ecc71']

for i, col in enumerate(comparison_cols):
    axes[i].bar(comparison.index, comparison[col], color=colors)
    axes[i].set_title(col, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)
    for j, v in enumerate(comparison[col]):
        axes[i].text(j, v + v*0.01, f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle('Clinical Profile Comparison Across Readmission Categories',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Part 8: Discharge Destination Analysis
Understanding if where patients go after discharge affects their probability of returning.

In [ ]:
discharge_readmit = df.groupby('Discharge Disposition')['Readmitted Under 30 Days'].agg(['mean', 'count']).reset_index()
discharge_readmit.columns = ['Discharge Disposition', 'Readmit Rate', 'Count']
discharge_readmit['Readmit Rate'] = discharge_readmit['Readmit Rate'] * 100
discharge_readmit = discharge_readmit[discharge_readmit['Count'] > 200].sort_values('Readmit Rate', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(discharge_readmit['Discharge Disposition'], discharge_readmit['Readmit Rate'],
               color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(discharge_readmit))))
ax.set_xlabel('30-Day Readmission Rate (%)')
ax.set_title('Discharge Destination vs 30-Day Readmission Rate', fontsize=14, fontweight='bold')
for bar, count in zip(bars, discharge_readmit['Count']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}% (n={count:,})', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## EDA Summary & Key Findings

| # | Finding | Business Implication |
| :--- | :--- | :--- |
| 1 | 30-day readmission rate is approximately 11.2% | Significant cost burden; intervention programs targeting even a small reduction can save millions |
| 2 | Prior inpatient visits is the strongest predictor of readmission | Patients with recurrent hospitalizations need dedicated post-discharge care coordinators |
| 3 | Emergency admissions have higher readmission rates than elective ones | Emergency protocols should include readmission risk screening at discharge |
| 4 | Patients with medication changes show different readmission patterns | Medication reconciliation programs at discharge can reduce confusion and non-compliance |
| 5 | Circulatory and Diabetes diagnoses dominate the readmission pool | Specialized chronic disease management pathways can target the highest-volume groups |
| 6 | Length of stay shows a non-linear relationship with readmission | Both very short and very long stays correlate with higher readmission — optimal stay windows should be studied |
| 7 | Insulin dosage changes correlate with readmission outcomes | Insulin management protocols should be reviewed for high-risk cohorts |
| 8 | Discharge to home vs. skilled nursing has different readmission profiles | Post-discharge support should vary based on destination |